In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.linear_model import LinearRegression
import numpy as np

have to do simple linear reg,multiple l reg ,lassocv and comapare the coefficient values

In [2]:
data = {
    "Study_Hours": [1, 2, 2, 3, 3, 4, 5, 5, 6, 7, 7, 8, 8, 9, 10],
    "Attendence": [55, 60, 65, 62, 70, 72, 75, 80, 78, 82, 85, 88, 90, 92, 95],
    "Assignment_Score": [40, 45, 50, 48, 55, 58, 62, 67, 65, 70, 74, 78, 80, 85, 90],
    "Student_ID": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115],
    "Final_Marks": [35, 40, 43, 45, 50, 54, 60, 64, 66, 71, 74, 79, 82, 87, 92]
}


In [3]:
df = pd.DataFrame(data)

X = df[
    [
        "Study_Hours",
        "Attendence",
        "Assignment_Score",
        "Student_ID"
    ]
]

y = df["Final_Marks"]


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (12, 4)
Testing data: (3, 4)


SIMPLE LINEAR REGRESSION

In [8]:
X_simple = df[["Study_Hours"]]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple,
    y,
    test_size=0.2,
    random_state=42
)

simple_model = LinearRegression()

simple_model.fit(X_train_s, y_train_s)

y_pred_simple = simple_model.predict(X_test_s)

In [9]:
simple_mae = mean_absolute_error(y_test_s, y_pred_simple)
simple_mse = mean_squared_error(y_test_s, y_pred_simple)
simple_rmse = np.sqrt(simple_mse)
simple_r2 = r2_score(y_test_s, y_pred_simple)

print("===== SIMPLE LINEAR REGRESSION =====")
print("Coefficient:", simple_model.coef_[0])
print("Intercept:", simple_model.intercept_)
print("MAE:", simple_mae)
print("MSE:", simple_mse)
print("RMSE:", simple_rmse)
print("R2 Score:", simple_r2)

===== SIMPLE LINEAR REGRESSION =====
Coefficient: 6.4793388429752055
Intercept: 28.52685950413224
MAE: 1.4166666666666667
MSE: 3.387056610431884
RMSE: 1.8403957754874043
R2 Score: 0.9907513624108353


MULTIPLE LINEAR REGRESSION

In [10]:
multiple_model = LinearRegression()

multiple_model.fit(X_train, y_train)

y_pred_multiple = multiple_model.predict(X_test)

In [11]:
multiple_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": multiple_model.coef_
})

print("===== MULTIPLE LINEAR REGRESSION =====")
print(multiple_coefficients)
print("Intercept:", multiple_model.intercept_)

===== MULTIPLE LINEAR REGRESSION =====
            Feature  Coefficient
0       Study_Hours     2.527015
1        Attendence     0.322483
2  Assignment_Score     0.227662
3        Student_ID     0.845633
Intercept: -81.4031661156238


In [12]:
multiple_mae = mean_absolute_error(y_test, y_pred_multiple)
multiple_mse = mean_squared_error(y_test, y_pred_multiple)
multiple_rmse = np.sqrt(multiple_mse)
multiple_r2 = r2_score(y_test, y_pred_multiple)

print("MAE:", multiple_mae)
print("MSE:", multiple_mse)
print("RMSE:", multiple_rmse)
print("R2 Score:", multiple_r2)

MAE: 0.9898930115061452
MSE: 1.1811493525507604
RMSE: 1.0868069527523094
R2 Score: 0.9967747742193699


Lasso Regression

In [13]:
lasso_model = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.1)
)

lasso_model.fit(X_train, y_train)

y_pred_lasso = lasso_model.predict(X_test)

In [14]:
lasso = lasso_model.named_steps["lasso"]

lasso_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Lasso Coefficient": lasso.coef_
})

print("===== LASSO REGRESSION =====")
print(lasso_coefficients)

===== LASSO REGRESSION =====
            Feature  Lasso Coefficient
0       Study_Hours           6.557418
1        Attendence           3.645076
2  Assignment_Score           3.222996
3        Student_ID           3.434937


In [15]:
lasso_mae = mean_absolute_error(y_test, y_pred_lasso)
lasso_mse = mean_squared_error(y_test, y_pred_lasso)
lasso_rmse = np.sqrt(lasso_mse)
lasso_r2 = r2_score(y_test, y_pred_lasso)

print("MAE:", lasso_mae)
print("MSE:", lasso_mse)
print("RMSE:", lasso_rmse)
print("R2 Score:", lasso_r2)

MAE: 0.8883785820961189
MSE: 0.9516530369875508
RMSE: 0.9755270559997559
R2 Score: 0.9974014328480316


LassoCV

In [17]:
from sklearn.linear_model import LassoCV

lasso_cv_model = make_pipeline(
    StandardScaler(),
    LassoCV(
        alphas=[0.001, 0.01, 0.1, 1, 10],
        cv=5,
        max_iter=10000
    )
)

lasso_cv_model.fit(X_train, y_train)

y_pred_lasso_cv = lasso_cv_model.predict(X_test)

lasso_cv = lasso_cv_model.named_steps["lassocv"]

best_alpha = lasso_cv.alpha_

print("===== LASSO CV =====")
print("Best Alpha:", best_alpha)

===== LASSO CV =====
Best Alpha: 0.01


In [18]:
lasso_cv_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "LassoCV Coefficient": lasso_cv.coef_
})

print(lasso_cv_coefficients)

            Feature  LassoCV Coefficient
0       Study_Hours             6.677354
1        Attendence             3.775885
2  Assignment_Score             3.136419
3        Student_ID             3.362363


In [19]:
lasso_cv_mae = mean_absolute_error(y_test, y_pred_lasso_cv)
lasso_cv_mse = mean_squared_error(y_test, y_pred_lasso_cv)
lasso_cv_rmse = np.sqrt(lasso_cv_mse)
lasso_cv_r2 = r2_score(y_test, y_pred_lasso_cv)

print("MAE:", lasso_cv_mae)
print("MSE:", lasso_cv_mse)
print("RMSE:", lasso_cv_rmse)
print("R2 Score:", lasso_cv_r2)

MAE: 1.0065255622070168
MSE: 1.2123274097690464
RMSE: 1.1010574053013977
R2 Score: 0.9966896399611889


compare all 

In [20]:
comparison = pd.DataFrame({
    "Model": [
        "Simple Linear Regression",
        "Multiple Linear Regression",
        "Lasso Regression",
        "LassoCV"
    ],
    
    "MAE": [
        simple_mae,
        multiple_mae,
        lasso_mae,
        lasso_cv_mae
    ],
    
    "MSE": [
        simple_mse,
        multiple_mse,
        lasso_mse,
        lasso_cv_mse
    ],
    
    "RMSE": [
        simple_rmse,
        multiple_rmse,
        lasso_rmse,
        lasso_cv_rmse
    ],
    
    "R2 Score": [
        simple_r2,
        multiple_r2,
        lasso_r2,
        lasso_cv_r2
    ]
})

comparison

,Model,MAE,MSE,RMSE,R2 Score
0,Simple Linear Regression,1.416667,3.387057,1.840396,0.990751
1,Multiple Linear Regression,0.989893,1.181149,1.086807,0.996775
2,Lasso Regression,0.888379,0.951653,0.975527,0.997401
3,LassoCV,1.006526,1.212327,1.101057,0.996690


In [21]:
# Get scaler and Lasso model from the pipeline
scaler = lasso_model.named_steps["standardscaler"]
model = lasso_model.named_steps["lasso"]

# Convert coefficients back to original units
original_coefficient = model.coef_ / scaler.scale_

# Calculate intercept in original units
original_intercept = (
    model.intercept_
    - np.sum(original_coefficient * scaler.mean_ / scaler.scale_)
)

print("Original coefficients:")
for feature, coefficient in zip(X.columns, original_coefficient):
    print(feature, ":", coefficient)

print("Original intercept:", original_intercept)

Original coefficients:
Study_Hours : 2.529160690473056
Attendence : 0.3230245875158471
Assignment_Score : 0.2275725434283605
Student_ID : 0.8193206470101531
Original intercept: 33.510786952991396
